# German cells: error against adaptability

Whether a finetuned "improvement" is real adaptation or a relocated center.
Per question, model/arm and mode, restricted to the German subpopulation
cells: mean prediction error `E = mean_i nEMD(WVS_i, LLM_i)`, dispersion
`D = median pairwise nEMD` within the survey cells and within the model's,
the adaptability ratio `AR = D_LLM / D_WVS`, a German-only center fit
`log1p(nEMD(WVS_i, LLM_i)) ~ log1p(nEMD(WVS_i, LLM_center))`, and two center
relocations: `C = nEMD(mean_i WVS_i, mean_i LLM_i)` over the German cells —
how far the model's German-average answer sits from the survey's — and
`C_all`, the same quantity over every retained cell of the run, how far the
model's overall answer sits from the whole survey's.

`AR = 1` means the model spreads its subpopulation answers as much as the
survey's own cells do; `AR -> 0` means one answer wearing 144 names. The
ideal corner is `E -> 0`, `AR -> 1`.

| encoding | meaning |
|---|---|
| colour + marker | the model |
| fill | hollow = untuned `base`, solid = `german` LoRA |
| centre dot | rides on every solid `german` marker |
| pale companion + thin rule | the same `german` arm at `x = C` instead of `x = E` |
| x | E, mean nEMD to the German WVS cells |
| y | AR, model dispersion over survey dispersion |

A second plate puts the two relocations against each other, `x = C` against
`y = C_all`, with the diagonal as the indifference rule: above it a model's
centre sits closer to the pooled German respondents than to the pooled
world, below it the reverse — where an untuned base is expected to live.

Recomputed from the run distributions, not the derived CSVs: the shipped
`pairwise_dispersion.csv` pools all 687 cells, and pooling across countries
is exactly what this notebook must not do — `C_all` is the one deliberate
exception, pooled on purpose to be the world-centre yardstick. Writes
`outputs/culture/german_adaptability.csv`, carrying `c_center_nemd` and
`c_center_all_nemd`, plus `figures/fig_german_adaptability_{ntp,fa}.{png,pdf}`
and `figures/fig_german_center_{ntp,fa}.{png,pdf}`. A companion notebook,
`german_structure.ipynb`, reads the CSV back in and asks a third, orthogonal
question over the same cells — not how much a model disperses but whether it
disperses along the real German social axes — so it must be run after this
one, not instead of it.

`.venv` has no jupyter and `uv run` must not be used here — it destroys the
hand-built sm_120 `llama-cpp-python` wheel. Run the code cells through
`.venv/bin/python`. Committed without cell outputs.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from matplotlib.figure import Figure
from matplotlib.lines import Line2D

from culture.palette import (
    MIXTRAL_ARCHIVED,
    MIXTRAL_FRESH,
    REFERENCE_MARKERS,
    REFERENCE_TONES,
    arm_order,
    model_marker,
    model_tone,
)
from culture.registry import ARMS, CULTURE_MODELS, csv_stems, is_base
from culture.tables import read_csv
from machine_bias_reproduction.analysis import load_source
from machine_bias_reproduction.config import FIGURES_ROOT, OUTPUTS_ROOT, paths_for
from machine_bias_reproduction.data import country_of
from machine_bias_reproduction.figures import save_plate
from machine_bias_reproduction.io_utils import write_csv
from machine_bias_reproduction.metrics import nemd, pairwise_nemd
from machine_bias_reproduction.plates import GRID, MUTED_INK
from machine_bias_reproduction.questions import resolve_questions
from machine_bias_reproduction.r_rng import center_holdout_indices

MODES = ("ntp", "fa")
MODEL_INDEX = {key: index for index, key in enumerate(CULTURE_MODELS)}
TABLE = OUTPUTS_ROOT / "culture" / "german_adaptability.csv"
XLABEL = "Mean nEMD to the German WVS cells (E)"
YLABEL = "Model dispersion / survey dispersion (AR)"

## Finding the runs

Same discovery as the coefficients notebook, but this one needs the response
distributions rather than the derived tables, so each run is loaded through
`load_source` and a missing or incomplete run raises instead of returning
`None` — the `except` is the whole skip-what-was-never-run mechanism. The
German cells are selected with the same `country_of` extractor the pipeline
itself groups by.

In [ ]:
def sources():
    found = [
        (f"{model.label} ({arm})", f"culture/{key}/{arm}", key, model.label, arm)
        for key, model in CULTURE_MODELS.items()
        for arm in arm_order(ARMS)
    ]
    found.append((MIXTRAL_ARCHIVED, "archived", None, MIXTRAL_ARCHIVED, None))
    found.append((MIXTRAL_FRESH, "fresh", None, MIXTRAL_FRESH, None))
    return found


def load_run(source, key, arm, question):
    stems = csv_stems(key, arm, question) if key else None
    try:
        return load_source(source, paths_for(source, question.var), question, stems)
    except (FileNotFoundError, ValueError):
        return None


def german_names(prepared):
    labels = prepared.names.to_series(index=prepared.names)
    return prepared.names[(country_of(labels) == "Germany").to_numpy()]

## The metrics, and the center fit

`E` is the mean per-cell nEMD, `D` the median pairwise nEMD over the same
cells — the dispersion *Machine Bias* Figure 4 shows visually — and `AR`
their ratio, model over survey.

The center fit replays the paper's center regression on the German cells
alone: `center_holdout_indices` draws the same R-compatible 20-cell holdout
over the German index range, the center is the mean of the 20 held-out model
distributions, and the OLS runs on the remaining cells. Same estimator as the
`center` row of `regression_fit.csv`, so `r2_center_german` reads directly
against the all-country `r2_center_reported` beside it. A center averaged
over every cell would share data with each response point and flatter the
very artifact under test.

In [ ]:
def center_r2(wvs, llm, mode):
    count = len(wvs)
    if count <= 21:
        return float("nan")
    indices = center_holdout_indices(count, mode)
    center = llm[indices].mean(axis=0)
    keep = np.ones(count, dtype=bool)
    keep[indices] = False
    response = pd.Series(np.log1p(nemd(wvs, llm))[keep], name="nEMD_log")
    frame = pd.DataFrame({"nEMD_center": np.log1p(nemd(wvs, center))[keep]})
    fit = sm.OLS(response, sm.add_constant(frame, has_constant="add")).fit()
    return float(fit.rsquared_adj)


def german_metrics(prepared, mode):
    names = german_names(prepared)
    columns = list(prepared.question.answer_columns)
    props = prepared.ntp_props if mode == "ntp" else prepared.fa_props
    wvs = prepared.wvs_props.loc[names, columns].to_numpy(dtype=np.float64)
    llm = props.loc[names, columns].to_numpy(dtype=np.float64)
    wvs_all = prepared.wvs_props.loc[prepared.names, columns].to_numpy(dtype=np.float64)
    llm_all = props.loc[prepared.names, columns].to_numpy(dtype=np.float64)
    d_wvs = float(np.median(pairwise_nemd(wvs)))
    d_llm = float(np.median(pairwise_nemd(llm)))
    return {
        "n_cells": int(len(names)),
        "e_mean_nemd": float(np.mean(nemd(wvs, llm))),
        "c_center_nemd": float(nemd(wvs.mean(axis=0), llm.mean(axis=0))),
        "c_center_all_nemd": float(nemd(wvs_all.mean(axis=0), llm_all.mean(axis=0))),
        "d_wvs": d_wvs,
        "d_llm": d_llm,
        "adaptability_ratio": d_llm / d_wvs,
        "r2_center_german": center_r2(wvs, llm, mode),
    }


def reported_center(outputs):
    fit = read_csv(outputs / "regression_fit.csv")
    if fit is None:
        return {}
    center = fit[fit["model"] == "center"]
    return dict(zip(center["mode"], center["adjusted_r_squared"], strict=True))

## Build

One row per run and mode. Three checks ride along: a complete run retains
exactly 144 German cells; every complete run of a question sees the same
survey dispersion; and the recomputed `E` agrees with the Germany-filtered
mean of that run's own `subpopulation_distances.csv`.

In [ ]:
def check(table):
    full = table[table["n_cells"] == 144]
    assert not full.empty
    assert table["n_cells"].le(144).all()
    for _, group in full.groupby(["question", "mode"]):
        assert np.allclose(group["d_wvs"], group["d_wvs"].iloc[0])
    row = full.iloc[0]
    distances = read_csv(paths_for(row["source"], row["question"]).outputs / "subpopulation_distances.csv")
    if distances is None:
        return
    german = distances[
        (distances["method"] == row["mode"].upper())
        & distances["subpopulation"].str.startswith("Germany")
    ]
    assert np.isclose(german["nEMD"].mean(), row["e_mean_nemd"])


def build():
    rows = []
    for question in resolve_questions(None):
        for label, source, key, model_label, arm in sources():
            prepared = load_run(source, key, arm, question)
            if prepared is None:
                continue
            reported = reported_center(paths_for(source, question.var).outputs)
            for mode in MODES:
                rows.append(
                    {
                        "question": question.var,
                        "question_label": question.label,
                        "model_key": key,
                        "model_label": model_label,
                        "arm": arm,
                        "series": label,
                        "source": source,
                        "mode": mode,
                        **german_metrics(prepared, mode),
                        "r2_center_reported": reported.get(mode),
                    }
                )
    table = pd.DataFrame(rows)
    if not table.empty:
        check(table)
    return table

In [ ]:
table = build()
write_csv(table, TABLE)
print(len(table), "rows,", int(table["n_cells"].eq(144).sum()), "complete")

## The plates

One panel per question. The first plate is `x = E`, `y = AR`, the dashed rule
at `AR = 1` where the model matches human heterogeneity; every `german` arm
appears twice, solid with a centre dot at its per-cell error `E` and pale at
its pooled-centre distance `C`, the two joined by a thin rule so relocation
and per-cell error read as one gesture. The second plate is `x = C` against
`y = C_all` with the diagonal as the indifference rule between the German
pool and the pooled world. The corners of each first panel name the
readings; the styles are the ones every cross-model plate already wears.

In [ ]:
def series_style(label, model_key, arm):
    if label in REFERENCE_TONES:
        ink = REFERENCE_TONES[label].ink
        return {
            "marker": REFERENCE_MARKERS[label],
            "color": ink,
            "markerfacecolor": ink,
            "markeredgecolor": ink if label == MIXTRAL_ARCHIVED else MUTED_INK,
        }
    index = MODEL_INDEX[model_key]
    ink = model_tone(model_key, index).ink
    return {
        "marker": model_marker(model_key, index),
        "color": ink,
        "markerfacecolor": "none" if is_base(arm) else ink,
        "markeredgecolor": ink,
    }


def companion_style(model_key):
    index = MODEL_INDEX[model_key]
    tone = model_tone(model_key, index)
    return {
        "marker": model_marker(model_key, index),
        "color": tone.ink,
        "markerfacecolor": tone.fill,
        "markeredgecolor": tone.ink,
    }


def is_german_arm(row):
    return isinstance(row["model_key"], str) and not is_base(row["arm"])


def solid_dot(axis, x, y, color):
    axis.plot(x, y, linestyle="none", marker=".", markersize=3, color=color, zorder=4)


def series_handles(frame):
    handles = []
    for series in dict.fromkeys(frame["series"]):
        subset = frame[frame["series"] == series]
        style = series_style(series, subset["model_key"].iat[0], subset["arm"].iat[0])
        handles.append(Line2D([], [], linestyle="none", markersize=6, label=series, **style))
    return handles


CORNERS = (
    (0.02, 0.97, "left", "top", "Accurate but over-dispersed"),
    (0.02, 0.03, "left", "bottom", "Accurate but under-dispersed"),
    (0.98, 0.03, "right", "bottom", "Inaccurate and under-dispersed"),
    (0.98, 0.97, "right", "top", "Inaccurate and over-dispersed"),
)


def adaptability_plate(table, mode):
    frame = table[table["mode"] == mode]
    questions = list(dict.fromkeys(table["question"]))
    figure = Figure(figsize=(3.6 * len(questions), 4.2), layout="constrained")
    axes = figure.subplots(1, len(questions), sharey=True, squeeze=False)[0]
    top = max(1.08, float(frame["adaptability_ratio"].max()) * 1.12)
    for axis, question in zip(axes, questions, strict=True):
        subset = frame[frame["question"] == question]
        axis.axhline(1.0, linestyle=(0, (4, 2)), color=MUTED_INK, linewidth=0.9)
        axis.grid(color=GRID, linewidth=0.6)
        axis.set_axisbelow(True)
        for _, row in subset.iterrows():
            style = series_style(row["series"], row["model_key"], row["arm"])
            axis.plot(
                row["e_mean_nemd"],
                row["adaptability_ratio"],
                linestyle="none",
                markersize=6.5,
                markeredgewidth=1.1,
                zorder=3,
                **style,
            )
            if is_german_arm(row):
                pale = companion_style(row["model_key"])
                axis.plot(
                    [row["c_center_nemd"], row["e_mean_nemd"]],
                    [row["adaptability_ratio"], row["adaptability_ratio"]],
                    linestyle="-",
                    linewidth=0.8,
                    color=pale["color"],
                    alpha=0.5,
                    zorder=2,
                )
                axis.plot(
                    row["c_center_nemd"],
                    row["adaptability_ratio"],
                    linestyle="none",
                    markersize=6.5,
                    markeredgewidth=1.1,
                    zorder=3,
                    **pale,
                )
                solid_dot(axis, row["e_mean_nemd"], row["adaptability_ratio"], "white")
                solid_dot(axis, row["c_center_nemd"], row["adaptability_ratio"], pale["color"])
        axis.set_title(table.loc[table["question"] == question, "question_label"].iat[0])
        axis.set_xlim(left=0)
        axis.set_ylim(0, top)
        axis.set_xlabel(XLABEL if len(questions) == 1 else "E")
    axes[0].set_ylabel(YLABEL)
    for x, y, ha, va, text in CORNERS:
        axes[0].text(
            x, y, text,
            transform=axes[0].transAxes,
            ha=ha, va=va,
            fontsize=7, style="italic", color=MUTED_INK,
        )
    figure.supxlabel(XLABEL, fontsize=9)
    figure.suptitle(mode.upper())
    handles = series_handles(frame)
    handles.append(
        Line2D(
            [], [],
            linestyle="-",
            linewidth=0.8,
            color=MUTED_INK,
            marker="o",
            markersize=5,
            markerfacecolor="none",
            markeredgecolor=MUTED_INK,
            label="same german arm at x = C",
        )
    )
    handles.append(
        Line2D([], [], linestyle=(0, (4, 2)), color=MUTED_INK, label="survey dispersion (AR = 1)")
    )
    figure.legend(handles=handles, loc="outside right upper")
    return save_plate(figure, FIGURES_ROOT / f"fig_german_adaptability_{mode}")


def center_plate(table, mode):
    frame = table[table["mode"] == mode]
    questions = list(dict.fromkeys(table["question"]))
    figure = Figure(figsize=(3.6 * len(questions), 4.2), layout="constrained")
    axes = figure.subplots(1, len(questions), sharey=True, squeeze=False)[0]
    limit = float(max(frame["c_center_nemd"].max(), frame["c_center_all_nemd"].max())) * 1.12
    for axis, question in zip(axes, questions, strict=True):
        subset = frame[frame["question"] == question]
        axis.plot([0, limit], [0, limit], linestyle=(0, (4, 2)), color=MUTED_INK, linewidth=0.9)
        axis.grid(color=GRID, linewidth=0.6)
        axis.set_axisbelow(True)
        for _, row in subset.iterrows():
            style = series_style(row["series"], row["model_key"], row["arm"])
            axis.plot(
                row["c_center_nemd"],
                row["c_center_all_nemd"],
                linestyle="none",
                markersize=6.5,
                markeredgewidth=1.1,
                zorder=3,
                **style,
            )
            if is_german_arm(row):
                solid_dot(axis, row["c_center_nemd"], row["c_center_all_nemd"], "white")
        axis.set_title(table.loc[table["question"] == question, "question_label"].iat[0])
        axis.set_xlim(0, limit)
        axis.set_ylim(0, limit)
        axis.set_xlabel("C" if len(questions) > 1 else "C, nEMD to the pooled German cells")
    axes[0].set_ylabel("C_all, nEMD to the whole survey pooled")
    axes[0].text(
        0.03, 0.97, "closer to the German pool",
        transform=axes[0].transAxes, ha="left", va="top",
        fontsize=7, style="italic", color=MUTED_INK,
    )
    axes[0].text(
        0.97, 0.03, "closer to the pooled world",
        transform=axes[0].transAxes, ha="right", va="bottom",
        fontsize=7, style="italic", color=MUTED_INK,
    )
    figure.supxlabel("C, nEMD to the pooled German cells", fontsize=9)
    figure.suptitle(mode.upper())
    handles = series_handles(frame)
    handles.append(
        Line2D([], [], linestyle=(0, (4, 2)), color=MUTED_INK, label="indifference (C = C_all)")
    )
    figure.legend(handles=handles, loc="outside right upper")
    return save_plate(figure, FIGURES_ROOT / f"fig_german_center_{mode}")

In [ ]:
files = []
for mode in MODES:
    files += adaptability_plate(table, mode)
    files += center_plate(table, mode)
print(len(files), "files")

## Reading the plates

- Hollow and solid separating horizontally in one colour while both hug the
  floor: the LoRA moved the model's center, not its social range — the 2D MDS
  "improvement" as pure relocation.
- A solid marker climbing toward the dashed rule while staying left of its
  hollow base is genuine adaptation, moving toward the ideal corner
  (`E -> 0`, `AR -> 1`) rather than deeper into either "inaccurate" corner.
- The pale companion left of its solid partner says the arm's pooled German
  answer sits closer to Germany's pooled answer than its per-cell error
  suggests — a relocated centre wearing per-cell noise. Companion and solid
  coinciding says the error is already pure centre distance.
- On the centre plate, a solid marker above the diagonal answered Germany's
  pooled cells more closely than the whole survey's; a hollow base marker
  below the diagonal is the world-centred prior the fine-tuning had to move.
- `r2_center_german` rising from base to arm while AR falls says one
  relocated point explains everything; it is in the CSV, not on the plates.
- The survey dispersion `d_wvs` is the same number for every complete run of
  a question — it is the yardstick, not a result.
- A run that dropped German cells is measured on its retained subset; the
  marker does not say so, `n_cells` in the CSV does. `c_center_all_nemd`
  pools whatever cells the run retained, all countries included, on purpose.
- Absent runs contribute no marker, not a marker at zero.
- Whether a high-`AR` marker's extra spread runs along real German social
  divides, or is socially arbitrary, is not something these plates can say —
  see `german_structure.ipynb` and its `ρ_structure` column for that
  question.